# CP2 NLP Operator Notebook — Kelompok 7

**Status:** supervised operator runbook. Tidak ada model training yang berjalan saat notebook ini dibuka. Jalankan dari root repository yang mengandung `src/cp2`. Jaga semua data, checkpoint, dan prediksi per-ID dalam folder privat. *Official test* tidak digunakan.

Akses JupyterLab menggunakan login institusi sendiri; jangan memasukkan password, token GitHub, atau token Hugging Face ke notebook.

In [ ]:
from pathlib import Path
import os, subprocess, sys
print('CWD:', Path.cwd())
print('Python:', sys.version.split()[0])
print('Repository root?', Path('src/cp2/nlp_runner.py').exists())
assert Path('src/cp2/nlp_runner.py').exists(), 'Pindah ke root repository dahulu'


## 1. Preflight
Pengecekan read-only. Bandingkan VRAM/RAM/disk dengan kebutuhan GPU sebelum menjalankan pelatihan.

In [ ]:
subprocess.run(['bash','scripts/cp2_ai_center_run.sh','preflight'],check=True)

## 2. Unit test (tanpa dataset)
Hasil lulus hanya mengonfirmasi kontrak sintetis, bukan model pretrained/real-data.

In [ ]:
subprocess.run(['bash','scripts/cp2_ai_center_run.sh','test'],check=True)

## 3. Inisialisasi & audit dataset
Membutuhkan akses ke sumber dataset pinned (atau cache metadata dan manifest CP1 yang sudah disalin). Hash/ukuran harus cocok. *Jangan* mengubah split.

In [ ]:
RUN_DATA_AUDIT = False  # ubah True hanya setelah izin akses dan workspace privat disiapkan
if RUN_DATA_AUDIT:
    for task in ['init-data','audit']:
        subprocess.run(['bash','scripts/cp2_ai_center_run.sh',task],check=True)
else:
    print('SKIPPED: data init/audit')

## 4. Kalibrasi komponen
Hanya jalankan ketika GPU bebas dan alokasi diizinkan. MHA sintetis bukan kalibrasi multimodal penuh. Simpan hasil di folder private.

In [ ]:
RUN_GPU_CALIBRATION = False
if RUN_GPU_CALIBRATION:
    for task in ['calibrate-attention','calibrate-b0','calibrate-vision']:
        subprocess.run(['bash','scripts/cp2_ai_center_run.sh',task],check=True)
else:
    print('SKIPPED: GPU calibration')

## 5. Smoke/Pilot B0
Jalankan setelah data audit lulus, VRAM aman, dan protokol dibekukan. Kedua mode bukan hasil riset CP2.

In [ ]:
RUN_SMOKE = False
RUN_PILOT = False
if RUN_SMOKE:
    subprocess.run(['bash','scripts/cp2_ai_center_run.sh','smoke-b0'],check=True)
if RUN_PILOT:
    subprocess.run(['bash','scripts/cp2_ai_center_run.sh','pilot-b0'],check=True)
if not (RUN_SMOKE or RUN_PILOT): print('SKIPPED: training')

## 6. Full B0 hanya dengan izin tim
Sengaja tidak ada sel yang otomatis menjalankan full training. Jalankan di terminal: `export CP2_CONFIRM_FULL=YES` lalu `bash scripts/cp2_ai_center_run.sh full-b0` setelah freeze dan audit.